# 주제 ① 사출성형 — 데이터 품질 진단 (01_data_quality)

**목적**: 주제 선정을 위해 데이터 특성과 필요한 전처리(결측·중복·이상치·불균형·누수)를 파악한다.
심사기준 1번 "데이터 이해 및 진단"에 대응. 모델링은 하지 않는다.

**데이터**: `data/1. 사출성형기 AI 데이터셋.zip` → `data_raw/` (CSV 4개, 가이드북 없음)

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import data_quality as dq
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 60)
def save(name): plt.tight_layout(); plt.savefig(FIG / name, dpi=110); plt.close(); print("saved", name)

## 0단계. 데이터 구성 파악

In [2]:
inv = dq.file_inventory(); inv

,key,file,size_MB,encoding
0,labeled_cn7,moldset_labeled_cn7.csv,0.56,utf-8
1,labeled_rg3,moldset_labeled_rg3.csv,0.55,utf-8
2,unlabeled_cn7,moldset_unlabeled_cn7.csv,16.64,utf-8
3,unlabeled_rg3,moldset_unlabeled_rg3.csv,16.98,utf-8


In [3]:
dfs = dq.load_all()
for k, d in dfs.items():
    print(f"{k:14s} shape={d.shape}  index[{d.index.min()}..{d.index.max()}] unique={d.index.is_unique} monotonic={d.index.is_monotonic_increasing}")
print("\n컬럼(labeled):", list(dfs["labeled_cn7"].columns))
print("labeled와 unlabeled 컬럼 차이:", set(dfs["labeled_cn7"].columns) ^ set(dfs["unlabeled_cn7"].columns))

labeled_cn7    shape=(1211, 25)  index[0..1210] unique=True monotonic=True
labeled_rg3    shape=(1182, 25)  index[1211..2392] unique=True monotonic=True
unlabeled_cn7  shape=(35239, 24)  index[114610..777220] unique=True monotonic=True
unlabeled_rg3  shape=(35941, 24)  index[119515..795309] unique=True monotonic=True

컬럼(labeled): ['PassOrFail', 'Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']
labeled와 unlabeled 컬럼 차이: {'PassOrFail'}


In [4]:
# 값이 이미 표준화되어 있는지 확인 (파일별 mean≈0, std≈1)
pd.DataFrame({k: {"mean_abs_max": d.drop(columns="PassOrFail", errors="ignore").mean().abs().max().round(6),
                  "std_min": d.drop(columns="PassOrFail", errors="ignore").std().min().round(4),
                  "std_max": d.drop(columns="PassOrFail", errors="ignore").std().max().round(4)}
              for k, d in dfs.items()}).T

,mean_abs_max,std_min,std_max
labeled_cn7,0.0,0.0,1.0004
labeled_rg3,0.0,0.0,1.0004
unlabeled_cn7,0.0,1.0,1.0000
unlabeled_rg3,0.0,1.0,1.0000


**0단계 결론**
- CSV 4개, 모두 UTF-8, 결측 없음. 가이드북/변수 설명서 없음 → 변수 의미는 컬럼명(사출성형기 표준 항목)으로 해석.
- `labeled_*` (약 1,200행, `PassOrFail` 라벨 있음) / `unlabeled_*` (약 35,000행, 라벨 없음). `cn7`, `rg3` 두 금형(제품) 세트.
- **모든 수치가 파일별로 독립 표준화(z-score)** 되어 있다. mean≈0, std≈1 (단, `Clamp_Open_Position`은 labeled에서 상수 0 → 원본도 상수였던 것으로 추정).
  → 원본 단위(초, bar, ℃)를 잃었고, labeled와 unlabeled의 스케일이 서로 다르다. 두 파일을 그대로 합쳐 쓰면 안 된다.
- 첫 컬럼은 이름 없는 **원본 행 인덱스**. labeled는 0~2392 연속(cn7 0~1210, rg3 1211~2392), unlabeled는 11만~79만 사이 띄엄띄엄 → 원래 더 큰 스트림에서 추출. **시간·설비·제품·로트 컬럼은 없다.**
- 학습/테스트 분할은 제공되지 않음. 테스트 데이터는 대회 측이 별도 제공하거나 unlabeled가 그 역할일 가능성 (추정).

## 1단계. 데이터 품질 진단
### 1-1. 기본 구조 (dtype, 고유값, 상수 컬럼)

In [5]:
for k in ["labeled_cn7", "labeled_rg3"]:
    print(f"===== {k}"); display(dq.basic_structure(dfs[k]))

===== labeled_cn7


,dtype,nunique,n_missing,min,max,flag
PassOrFail,int64,2,0,0.000,1.000,NEAR_CONST
Injection_Time,float64,15,0,-7.549,2.080,
Filling_Time,float64,14,0,-7.777,2.297,
Plasticizing_Time,float64,34,0,-0.723,20.038,
Cycle_Time,float64,13,0,-3.706,12.562,
Clamp_Close_Time,float64,3,0,-0.902,2.860,NEAR_CONST
Cushion_Position,float64,5,0,-2.613,6.944,NEAR_CONST
Plasticizing_Position,float64,13,0,-2.877,1.592,
Clamp_Open_Position,float64,1,0,0.000,0.000,CONSTANT
Max_Injection_Speed,float64,20,0,-1.542,9.556,


===== labeled_rg3


,dtype,nunique,n_missing,min,max,flag
PassOrFail,int64,2,0,0.000,1.000,NEAR_CONST
Injection_Time,float64,3,0,-8.628,8.570,NEAR_CONST
Filling_Time,float64,2,0,-1.193,0.839,NEAR_CONST
Plasticizing_Time,float64,27,0,-2.736,3.774,
Cycle_Time,float64,10,0,-1.395,11.762,
Clamp_Close_Time,float64,3,0,-1.807,1.033,NEAR_CONST
Cushion_Position,float64,7,0,-1.912,2.868,
Plasticizing_Position,float64,8,0,-2.537,2.933,
Clamp_Open_Position,float64,1,0,0.000,0.000,CONSTANT
Max_Injection_Speed,float64,7,0,-2.522,2.188,


In [6]:
for k in ["unlabeled_cn7", "unlabeled_rg3"]:
    s = dq.basic_structure(dfs[k]); print(f"===== {k}: constant/near-const =", s[s.flag != ""].index.tolist()); display(s[["nunique","min","max"]].T)

===== unlabeled_cn7: constant/near-const = []


,Injection_Time,Filling_Time,Plasticizing_Time,Cycle_Time,Clamp_Close_Time,Cushion_Position,Plasticizing_Position,Clamp_Open_Position,Max_Injection_Speed,Max_Screw_RPM,Average_Screw_RPM,Max_Injection_Pressure,Max_Switch_Over_Pressure,Max_Back_Pressure,Average_Back_Pressure,Barrel_Temperature_1,Barrel_Temperature_2,Barrel_Temperature_3,Barrel_Temperature_4,Barrel_Temperature_5,Barrel_Temperature_6,Hopper_Temperature,Mold_Temperature_3,Mold_Temperature_4
nunique,115.000,124.000,307.000,251.00,69.000,85.000,169.000,47.000,242.000,21.000,50.000,179.000,254.000,184.000,116.000,87.000,76.000,82.000,128.000,95.000,81.000,279.000,182.000,267.000
min,-2.402,-7.252,-1.275,-3.77,-1.108,-0.975,-1.438,-4.626,-1.708,-0.968,-0.970,-1.118,-0.997,-1.216,-0.938,-0.969,-2.666,-5.175,-4.485,-3.765,-2.390,-4.496,-0.952,-0.936
max,1.726,1.973,3.385,2.09,1.114,1.036,1.555,0.862,2.980,3.115,1.562,2.289,1.253,4.607,2.629,1.066,1.092,1.057,2.058,2.410,1.485,1.774,2.046,2.298


===== unlabeled_rg3: constant/near-const = []


,Injection_Time,Filling_Time,Plasticizing_Time,Cycle_Time,Clamp_Close_Time,Cushion_Position,Plasticizing_Position,Clamp_Open_Position,Max_Injection_Speed,Max_Screw_RPM,Average_Screw_RPM,Max_Injection_Pressure,Max_Switch_Over_Pressure,Max_Back_Pressure,Average_Back_Pressure,Barrel_Temperature_1,Barrel_Temperature_2,Barrel_Temperature_3,Barrel_Temperature_4,Barrel_Temperature_5,Barrel_Temperature_6,Hopper_Temperature,Mold_Temperature_3,Mold_Temperature_4
nunique,91.000,97.000,470.000,371.000,88.000,68.000,207.000,53.000,179.000,28.000,83.000,236.000,303.000,233.000,208.000,78.000,93.000,91.000,144.000,106.000,89.000,329.000,156.000,258.000
min,-1.151,-1.308,-1.534,-4.268,-1.634,-1.376,-1.938,-0.818,-2.131,-1.179,-1.297,-3.555,-0.563,-1.381,-1.154,-1.311,-2.266,-3.300,-3.089,-3.056,-3.302,-2.479,-1.277,-1.225
max,2.801,1.663,4.639,1.745,5.977,0.762,3.086,1.916,0.965,2.273,1.296,1.821,121.252,2.555,2.961,0.824,1.017,0.922,0.921,0.892,1.131,1.330,1.756,2.185


### 1-2. 생산단위 식별
- 컬럼은 모두 **샷(1 사이클) 단위 요약값**(사이클타임, 최대 사출압, 배럴 온도 등)이다 → **1행 = 1샷** (추정, 근거: 변수 구성이 사출기 샷 데이터 로그의 표준 항목).
- 설비/금형/제품/시간/로트 컬럼이 **전혀 없음**. 유일한 계층 정보는 파일명(`cn7` vs `rg3` = 금형셋 2종)과 원본 행 인덱스(수집 순서 추정).
- 같은 X 벡터가 2번씩 나타나는 구조(아래 1-4)로 보아, 원본에서 한 샷이 2행으로 기록되었거나(예: 2캐비티·2개 검사 결과), 의도적으로 복제된 것으로 **추정**.

### 1-3. 결측치

In [7]:
pd.DataFrame({k: d.isna().sum().sum() for k, d in dfs.items()}, index=["n_missing"]).T

,n_missing
labeled_cn7,0
labeled_rg3,0
unlabeled_cn7,0
unlabeled_rg3,0


명시적 결측(NaN)은 **0개**. 그러나 unlabeled에는 여러 센서가 동시에 한 값에 고정된 "비가동 블록"이 있어(1-7 참고) 이것이 사실상의 **숨은 결측/무효 행**이다.

### 1-4. 중복 및 라벨 충돌

In [8]:
dup = pd.DataFrame({k: dq.duplicate_report(dfs[k]) for k in dfs}).T
dup

,n_rows,full_dup_rows,feature_dup_rows,n_distinct_X,adjacent_identical_pairs,group_size_dist,conflict_groups,fails_in_conflict,total_fails,distinct_fail_X,both_fail_pairs
labeled_cn7,1211,594,605,606,570,"{1: 1, 2: 605}",11,11,17,14,3
labeled_rg3,1182,566,591,591,574,{2: 591},25,25,25,25,0
unlabeled_cn7,35239,6468,6468,28771,6467,"{1: 22303, 2: 6468}",NaN,NaN,NaN,NaN,NaN
unlabeled_rg3,35941,6234,6234,29707,6233,"{1: 23473, 2: 6234}",NaN,NaN,NaN,NaN,NaN


In [9]:
# labeled: 같은 X 쌍이 인접 행인지, 불량이 행 순서상 어디에 있는지
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=False)
for ax, k in zip(axes, ["labeled_cn7", "labeled_rg3"]):
    d = dfs[k]; y = d["PassOrFail"].values; pos = np.where(y == 1)[0]
    ax.vlines(pos, 0, 1, color="crimson"); ax.set_xlim(0, len(d)); ax.set_yticks([])
    ax.set_title(f"{k}: 불량(=1) 위치 (행 순서, n_fail={len(pos)}/{len(d)})")
save("fail_positions_labeled.png")
print("cn7 fail positions:", np.where(dfs['labeled_cn7'].PassOrFail.values==1)[0].tolist())
print("rg3 fail positions:", np.where(dfs['labeled_rg3'].PassOrFail.values==1)[0].tolist())

saved fail_positions_labeled.png
cn7 fail positions: [47, 80, 95, 99, 101, 103, 105, 107, 109, 111, 113, 115, 116, 117, 118, 119, 120]
rg3 fail positions: [136, 160, 170, 204, 244, 300, 304, 310, 322, 346, 407, 410, 463, 550, 620, 686, 726, 762, 810, 894, 952, 966, 972, 1120, 1132]


**중복 진단 결과 (핵심 이슈)**
- labeled: 모든 X 벡터가 **정확히 2회씩** 등장 (cn7 605쌍+1, rg3 591쌍). 실제 고유 샷은 **cn7 606개, rg3 591개**뿐. 쌍의 약 95%가 인접 행.
- **라벨 충돌**: 같은 X인데 한 행은 양품·한 행은 불량인 쌍이 cn7 11쌍, rg3 25쌍. 즉 **rg3 불량 25개는 전부, cn7 불량 17개 중 11개가 충돌 쌍**에 속한다.
  → 피처만으로는 이 불량들을 원리적으로 구분할 수 없다(동일 입력 → 다른 출력). 라벨 노이즈이거나, 캐비티별/검사자별 결과가 다른 것으로 추정.
- cn7 불량은 행 99~120 구간에 몰려 있음(첫 20% 구간 불량률 7%, 나머지 0%) → 특정 시점의 연속 불량 사건 1건에 가까움. rg3는 전 구간에 분산.
- unlabeled: 완전 중복 행 약 18%. 대부분 인접한 2행 반복(같은 기록 방식의 흔적).
- **검증 설계 함의**: 랜덤 split을 하면 같은 샷의 쌍이 train/test에 나뉘어 들어가 **누수**가 발생한다. 반드시 X-키 기준 GroupKFold 또는 쌍 제거 후 분할이 필요.

### 1-5. 이상치

In [10]:
for k in ["labeled_cn7", "labeled_rg3"]:
    X, y = dq.split_xy(dfs[k]); print(f"===== {k} (|z|>3 기준)"); display(dq.outlier_table(X, y).head(12))

===== labeled_cn7 (|z|>3 기준)


,n_outlier,nunique,min,max,n_iqr_outlier,fail_in_outlier
Barrel_Temperature_5,16,22,-4.61,3.39,44,0
Injection_Time,10,15,-7.55,2.08,10,6
Max_Injection_Speed,10,20,-1.54,9.56,12,6
Filling_Time,10,14,-7.78,2.30,10,6
Max_Injection_Pressure,10,8,-10.86,1.68,28,6
Cushion_Position,8,5,-2.61,6.94,2,0
Cycle_Time,6,13,-3.71,12.56,34,0
Plasticizing_Time,6,34,-0.72,20.04,10,0
Average_Screw_RPM,6,7,-1.29,16.09,308,0
Max_Screw_RPM,6,8,-2.49,3.43,6,0


===== labeled_rg3 (|z|>3 기준)


,n_outlier,nunique,min,max,n_iqr_outlier,fail_in_outlier
Injection_Time,16,3,-8.63,8.57,16,0
Barrel_Temperature_5,12,21,-4.42,4.36,22,0
Cycle_Time,8,10,-1.40,11.76,66,0
Plasticizing_Time,8,27,-2.74,3.77,22,0
Max_Switch_Over_Pressure,4,17,-13.79,1.55,4,0
Barrel_Temperature_2,4,22,-2.19,3.14,4,0
Max_Injection_Pressure,2,10,-3.39,2.52,16,0
Barrel_Temperature_1,2,25,-3.17,2.66,26,0
Hopper_Temperature,2,52,-3.07,2.12,24,0
Clamp_Open_Position,0,1,0.00,0.00,0,0


In [11]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, k in zip(axes, ["labeled_cn7", "labeled_rg3"]):
    X, y = dq.split_xy(dfs[k]); m = X.melt(var_name="var", value_name="z")
    sns.boxplot(data=m, x="var", y="z", ax=ax, fliersize=2, color="lightsteelblue")
    ax.axhline(3, ls="--", c="gray"); ax.axhline(-3, ls="--", c="gray"); ax.set_title(f"{k}: 변수별 표준화값 분포"); ax.tick_params(axis="x", rotation=70)
save("outlier_boxplot_labeled.png")

saved outlier_boxplot_labeled.png


In [12]:
# 극단 이상치와 불량의 관계: cn7에서 |z|>3 행 중 불량 비율
X, y = dq.split_xy(dfs["labeled_cn7"])
ext = (X.abs() > 3).any(axis=1)
print("cn7: 극단행 수", ext.sum(), "/ 그 중 불량", int(y[ext].sum()), "| 비극단행 불량", int(y[~ext].sum()))
X, y = dq.split_xy(dfs["labeled_rg3"]); ext = (X.abs() > 3).any(axis=1)
print("rg3: 극단행 수", ext.sum(), "/ 그 중 불량", int(y[ext].sum()), "| 비극단행 불량", int(y[~ext].sum()))

cn7: 극단행 수 48 / 그 중 불량 6 | 비극단행 불량 11
rg3: 극단행 수 56 / 그 중 불량 0 | 비극단행 불량 25


**이상치 진단 결과**
- 표준화 상태이므로 물리적 불가능값(음수 온도 등)은 직접 판단 불가. 대신 |z|가 매우 큰 값(cn7 `Plasticizing_Time` 20, `Average_Screw_RPM` 16, `Cycle_Time` 12.5; rg3 `Max_Switch_Over_Pressure` −13.8, unlabeled rg3 121)이 존재.
- cn7에서 `Injection_Time`, `Filling_Time`, `Max_Injection_Pressure`, `Max_Injection_Speed`가 동시에 극단값인 10행 중 **6행이 불량** → 이 극단값은 센서 오류가 아니라 **실제 공정 이상(충전 불량 등)** 일 가능성이 높다. 함부로 제거하면 불량 신호를 지운다.
- 반면 `Plasticizing_Time` 20σ, `Cycle_Time` 12σ 같은 단일 변수 극단값은 불량과 무관(양품) → 설비 정지/재가동 직후 샷 또는 센서 오류로 추정.
- **구분 기준 제안**: (a) 사출·충전 계열 변수가 **함께** 극단이면 공정 이상 후보(유지), (b) 한 변수만 단독 극단이고 라벨 양품이면 센서/로그 이상(윈저라이징 또는 플래그 변수화), (c) unlabeled의 고정값 블록은 비가동 행(제거).
- rg3의 `Injection_Time`(3개 값), `Filling_Time`(2개 값)은 사실상 이산 변수 → 원본이 셋포인트 값이거나 해상도가 낮은 것으로 추정.

### 1-6. 불균형

In [13]:
lab = pd.DataFrame({k: dfs[k]["PassOrFail"].value_counts() for k in ["labeled_cn7", "labeled_rg3"]}).T
lab["fail_rate"] = (lab[1] / lab.sum(axis=1)).round(4); lab["distinct_fail_X"] = [dq.duplicate_report(dfs[k])["distinct_fail_X"] for k in lab.index]
display(lab)
fig, ax = plt.subplots(figsize=(6, 3.5)); lab[[0, 1]].plot.bar(ax=ax, color=["steelblue", "crimson"]); ax.set_title("라벨 분포 (0=양품, 1=불량)"); ax.set_yscale("log")
save("label_imbalance.png")

PassOrFail,0,1,fail_rate,distinct_fail_X
labeled_cn7,1194,17,0.0140,14
labeled_rg3,1157,25,0.0212,25


saved label_imbalance.png


In [14]:
# 행 순서 5분위별 불량률 (시간 대용)
for k in ["labeled_cn7", "labeled_rg3"]:
    y = dfs[k]["PassOrFail"]; q = pd.qcut(np.arange(len(y)), 5, labels=[f"Q{i}" for i in range(1, 6)])
    print(k, y.groupby(q, observed=True).agg(["sum", "mean"]).round(4).T.to_dict())

labeled_cn7 {'Q1': {'sum': 17.0, 'mean': 0.07}, 'Q2': {'sum': 0.0, 'mean': 0.0}, 'Q3': {'sum': 0.0, 'mean': 0.0}, 'Q4': {'sum': 0.0, 'mean': 0.0}, 'Q5': {'sum': 0.0, 'mean': 0.0}}
labeled_rg3 {'Q1': {'sum': 4.0, 'mean': 0.0169}, 'Q2': {'sum': 9.0, 'mean': 0.0381}, 'Q3': {'sum': 3.0, 'mean': 0.0127}, 'Q4': {'sum': 4.0, 'mean': 0.0169}, 'Q5': {'sum': 5.0, 'mean': 0.0211}}


**불균형 진단 결과**
- 불량률 cn7 1.4%(17/1211), rg3 2.1%(25/1182). 중복 제거 후 고유 불량 X는 cn7 14개, rg3 25개, 그중 라벨이 모호하지 않은 것은 **cn7 3개, rg3 0개**.
- 불량 유형 구분 없음(이진). 설비/제품 구분은 파일 단위(2종)뿐.
- 극단적 소수 클래스 + 라벨 충돌 → F1 기준 평가 시 분산이 매우 큼. 반복 CV, 확률 보정, PR-AUC 병행이 필요.

### 1-7. 시간 구조 (행 순서·원본 인덱스를 시간 대용으로 사용)

In [15]:
# labeled: 행 순서에 따른 주요 변수 이동 평균 (drift)
cols = ["Cycle_Time", "Mold_Temperature_3", "Max_Injection_Pressure", "Barrel_Temperature_1"]
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
for ax, k in zip(axes, ["labeled_cn7", "labeled_rg3"]):
    X, y = dq.split_xy(dfs[k])
    for c in cols: ax.plot(X[c].rolling(50, center=True).mean().values, label=c)
    for p in np.where(y.values == 1)[0]: ax.axvline(p, color="crimson", alpha=.25)
    ax.set_title(f"{k}: 이동평균(50) — 빨간선=불량"); ax.legend(loc="upper right", fontsize=8)
save("drift_labeled.png")

saved drift_labeled.png


In [16]:
# unlabeled: 원본 인덱스 간격, 비가동 블록
for k in ["unlabeled_cn7", "unlabeled_rg3"]:
    U = dfs[k]; gp = np.diff(U.index); blk = dq.idle_block_mask(U); rl = dq.run_lengths(blk)
    print(f"{k}: 인덱스 간격 median={np.median(gp):.0f}, mean={gp.mean():.1f}, max={gp.max()}, gap==1 비율={(gp==1).mean():.2f}")
    print(f"   비가동 블록 행 {blk.sum()} ({blk.mean():.1%}), 블록 run 수 {len(rl)}, run 길이 median {rl.median():.0f} max {rl.max()}")
    print("   블록 내 고유값 1개 컬럼:", U[blk].nunique().pipe(lambda s: s[s == 1].index.tolist()))

unlabeled_cn7: 인덱스 간격 median=2, mean=18.8, max=181331, gap==1 비율=0.43
   비가동 블록 행 18151 (51.5%), 블록 run 수 1662, run 길이 median 2 max 3640
   블록 내 고유값 1개 컬럼: ['Max_Screw_RPM', 'Average_Screw_RPM', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Mold_Temperature_3', 'Mold_Temperature_4']
unlabeled_rg3: 인덱스 간격 median=2, mean=18.8, max=169361, gap==1 비율=0.37
   비가동 블록 행 13154 (36.6%), 블록 run 수 2287, run 길이 median 2 max 1829
   블록 내 고유값 1개 컬럼: ['Max_Screw_RPM', 'Average_Screw_RPM', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Mold_Temperature_3', 'Mold_Temperature_4']


In [17]:
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for i, k in enumerate(["unlabeled_cn7", "unlabeled_rg3"]):
    U = dfs[k]; blk = dq.idle_block_mask(U)
    axes[i, 0].hist(U["Barrel_Temperature_1"], bins=60, color="gray"); axes[i, 0].set_title(f"{k}: Barrel_Temperature_1 히스토그램 (왼쪽 스파이크=비가동 블록)")
    axes[i, 1].plot(blk.values[:4000].astype(int), lw=.5); axes[i, 1].set_title(f"{k}: 비가동 블록 마스크 (앞 4000행)"); axes[i, 1].set_yticks([0, 1])
save("unlabeled_idle_block.png")

saved unlabeled_idle_block.png


In [18]:
# 상관 구조: labeled vs unlabeled(전체) vs unlabeled(비가동 제거)
X, _ = dq.split_xy(dfs["labeled_cn7"]); U = dfs["unlabeled_cn7"]; Ub = U[~dq.idle_block_mask(U)]
print("labeled_cn7 |corr|>0.9 쌍:", len(dq.high_corr_pairs(X)))
print("unlabeled_cn7 전체 |corr|>0.9 쌍:", len(dq.high_corr_pairs(U)))
print("unlabeled_cn7 비가동 제거 후 |corr|>0.9 쌍:", len(dq.high_corr_pairs(Ub)))
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (t, D) in zip(axes, [("labeled_cn7", X), ("unlabeled_cn7 전체", U), ("unlabeled_cn7 비가동 제거", Ub)]):
    sns.heatmap(D.loc[:, D.nunique() > 1].corr(), vmin=-1, vmax=1, cmap="coolwarm", ax=ax, cbar=False); ax.set_title(t); ax.tick_params(labelsize=6)
save("corr_heatmaps_cn7.png")
print(dq.high_corr_pairs(X).round(3))

labeled_cn7 |corr|>0.9 쌍: 3
unlabeled_cn7 전체 |corr|>0.9 쌍: 101
unlabeled_cn7 비가동 제거 후 |corr|>0.9 쌍: 52


saved corr_heatmaps_cn7.png
Injection_Time      Filling_Time             0.980
Mold_Temperature_3  Mold_Temperature_4       0.978
Max_Back_Pressure   Average_Back_Pressure    0.940
dtype: float64


**시간 구조 진단 결과**
- 시간 컬럼이 없어 **행 순서·원본 인덱스를 시간 대용**으로 썼다(추정). labeled는 인덱스 연속, unlabeled는 간격 median 2·mean 19·최대 18만 → 큰 공백(설비 정지, 다른 제품 생산 등)이 여러 번 있다.
- labeled cn7은 `Cycle_Time`이 행 순서에 따라 단조 감소(+0.7σ → −0.7σ), `Mold_Temperature`도 큰 폭으로 이동 → **뚜렷한 drift**. 불량은 초반 한 구간에 집중.
  → 단일 변수 AUC가 0.89인 `Mold_Temperature_3/4`는 "불량 조건"이 아니라 "불량이 난 시기"를 가리킬 가능성이 큼(누수성 상관, 1-8).
- unlabeled에는 6개 센서가 한 값에 고정된 **비가동 블록**이 cn7 51.5%, rg3 36.6% 섞여 있고 짧은 run(median 2행)으로 반복 등장. 이 블록이 전 변수를 한 방향으로 끌어당겨 상관 0.9 이상 쌍이 100개 넘게 생긴다(허위 상관). 제거 후에도 배럴 온도 간 상관은 높게 남음(실제 물리적 연동).
- labeled에는 이런 블록이 없다 → labeled와 unlabeled는 **수집 조건(또는 전처리)이 다르다**.

### 1-8. 누수 위험

In [19]:
for k in ["labeled_cn7", "labeled_rg3"]:
    X, y = dq.split_xy(dfs[k]); a = dq.univariate_auc(X, y)
    md_ = (X[y == 1].mean() - X[y == 0].mean()).abs().round(2)
    print(f"===== {k}"); display(pd.DataFrame({"uni_AUC": a.round(3), "|mean_diff|(σ)": md_[a.index]}).head(10))

===== labeled_cn7


,uni_AUC,|mean_diff|(σ)
Mold_Temperature_3,0.894,1.23
Mold_Temperature_4,0.892,1.39
Plasticizing_Position,0.870,1.52
Max_Back_Pressure,0.852,1.50
Average_Back_Pressure,0.845,1.36
Plasticizing_Time,0.809,0.44
Clamp_Close_Time,0.791,1.22
Cushion_Position,0.767,1.29
Max_Switch_Over_Pressure,0.758,0.86
Hopper_Temperature,0.711,0.68


===== labeled_rg3


,uni_AUC,|mean_diff|(σ)
Barrel_Temperature_5,0.635,0.46
Plasticizing_Time,0.561,0.18
Barrel_Temperature_3,0.548,0.14
Cycle_Time,0.546,0.01
Clamp_Close_Time,0.546,0.19
Hopper_Temperature,0.545,0.14
Max_Screw_RPM,0.538,0.14
Average_Screw_RPM,0.530,0.14
Mold_Temperature_4,0.530,0.08
Mold_Temperature_3,0.529,0.09


**누수 진단 결과**
- 라벨과 1:1인 변수는 없음. 사후 정보(검사 결과 파생값) 컬럼도 없음.
- 그러나 (1) **중복 쌍 누수**: 같은 샷이 2행 → 랜덤 split 시 test 답이 train에 있음. (2) **시간 누수**: cn7에서 drift 변수(금형온도, 사이클타임)가 불량 시기와 겹쳐 높은 단일 AUC(0.87~0.89)를 보임. 이 변수를 랜덤 split으로 평가하면 성능이 과대평가되고, 미래 구간에서는 재현되지 않을 가능성이 큼.
- rg3는 단일 AUC 최대 0.64로 낮고, 불량 전부가 라벨 충돌 쌍 → **피처로 설명되지 않는 불량**. 모델 성능 한계가 데이터에서 이미 정해져 있다.

## 2단계. 진단 결과 → 필요 처리 정리

| 항목 | 필요도 | 근거 | 권장 방법 |
|---|---|---|---|
| 결측치 처리 | **하** (명시적) / **상** (숨은 결측) | NaN 0개. 그러나 unlabeled 비가동 블록 37~52%가 사실상 무효 행 | `idle_block_mask`로 블록 행 제거 또는 "비가동" 플래그 변수화. labeled에는 없음 |
| 중복 처리 | **상** | labeled 전 행이 2회 중복, 불량의 대부분이 라벨 충돌 쌍 | X-키 기준 그룹화 → 쌍 단위 집계(라벨 = max 또는 mean → "불량 의심 확률"). 랜덤 split 금지, **GroupKFold(X-키)** |
| 이상치 처리 | **중** | 극단값 존재하나 cn7에서는 충전 계열 극단값이 불량과 강하게 연결 | 라벨 확인 전 삭제 금지. 단독 변수 극단(20σ)은 클리핑(±5σ) + 플래그 변수. 다변수 동시 극단은 유지 |
| 불균형 처리 | **상** | 불량률 1.4~2.1%, 고유 불량 X 14/25개, 모호하지 않은 불량 3/0개 | class_weight·threshold 튜닝 우선. SMOTE는 고유 샘플이 너무 적어 위험. 반복 층화 GroupKFold + PR-AUC/F1 동시 보고, 확률 보정 |
| 도메인 지식 결합 | **상** | 원본 단위 소실, 변수 의미는 사출 표준 항목이라 파생 가능 | 파생 후보: `Filling/Injection` 비, `Cushion - Plasticizing_Position`(계량 안정성), 배럴 온도 구배(T1−T6, 인접차), `Max_Back − Avg_Back`, `Cycle − (Injection+Plasticizing)`(대기 시간), 이전 샷 대비 변화량(행 순서 기준). 스케일이 파일별로 달라 **파일 내부에서만** 계산 |
| 검증 전략 | **상** | 시간 컬럼 없음, drift 존재, 중복 쌍 | 1순위: **X-키 GroupKFold + 행 순서 기반 시간 블록 split** 둘 다 보고(랜덤 대비 성능 차이가 곧 누수 크기). cn7과 rg3는 **별도 모델 또는 금형 플래그** (스케일 상이). unlabeled는 자기지도/이상탐지 사전학습·드리프트 점검용으로만 사용, 성능 평가에는 사용 불가 |

### 주제 ① 관점 총평
- **강점**: 표 형태·소규모라 빠르게 돌릴 수 있고, 변수가 사출 공정 표준 항목이라 도메인 해석과 "FN/FP 집중 조건"(심사 3번) 설명이 쉬움. unlabeled 7만 행으로 이상탐지·반지도 등 차별화(심사 5번) 여지 있음.
- **리스크**: (1) 불량 표본이 사실상 3~14개 수준이고 대부분 라벨 충돌 → F1이 극단적으로 불안정, 모델 비교 결과의 신뢰도 낮음. (2) 시간·설비 정보 부재로 "시간·설비·제품 간 관계" 서술(심사 1번)이 추정에 그침. (3) 표준화로 원본 단위 손실 → 현장 활용안(심사 4번)의 임계값을 실제 단위로 제시 못 함.
- **난이도 한 줄 평**: 코드 난이도는 낮지만 **데이터 자체의 정보량이 적어 "높은 F1"보다 "왜 예측이 어려운지·라벨 충돌·검증 누수를 정직하게 진단"하는 스토리로 가야 하는 주제**. 데이터 진단(15점)·오류분석(15점)에서 점수를 벌고 모델(40점)에서는 안정적 비교 설계로 방어하는 전략이 맞다.